# 🧪 Pruebas Unitarias en Python con Mocks

Este notebook cubre técnicas de testing con `unittest.mock`, incluyendo cómo parchecar funciones y módulos correctamente.


In [ ]:
import unittest
from unittest.mock import patch, MagicMock
import datetime

## ❌ El Problema: Patching Incorrecto de `datetime.datetime`

El error ocurre porque cuando usas `@patch('datetime.datetime')`, **reemplazas TODO**, incluyendo el constructor. Esto significa:

1. `datetime.datetime(2024, 1, 1, 9, 0, 0)` crea un MagicMock, no un datetime real
2. Cuando el código llama a `datetime.datetime.now().hour`, obtiene un MagicMock en lugar de un entero
3. La comparación `MagicMock < 12` falla con TypeError

**Mensaje de error:**
```
TypeError: '<' not supported between instances of 'MagicMock' and 'int'
```


In [ ]:
# --- Código a probar ---
def obtener_saludo():
    """Retorna un saludo según la hora del día."""
    hora = datetime.datetime.now().hour
    if hora < 12:
        return "Buenos días"
    elif hora < 18:
        return "Buenas tardes"
    else:
        return "Buenas noches"


# ❌ VERSIÓN INCORRECTA - ESTO FALLARÁ
class TestSaludoIncorrecto(unittest.TestCase):

    @patch('datetime.datetime')  # ❌ Esto parcheaALL datetime.datetime
    def test_saludo_en_la_manana(self, mock_datetime):
        # El problema: datetime.datetime ya está parcheado
        # Entonces esto crea un MagicMock, no un datetime real
        mock_datetime.now.return_value = datetime.datetime(2024, 1, 1, 9, 0, 0)
        self.assertEqual(obtener_saludo(), "Buenos días")


print("=" * 60)
print("EJECUTANDO VERSIÓN INCORRECTA...")
print("=" * 60)
suite = unittest.TestLoader().loadTestsFromTestCase(TestSaludoIncorrecto)
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
print(f"\n✗ Errores: {resultado.errors}")


## ✅ SOLUCIÓN 1: Parchacar `datetime.datetime.now` Directamente

La mejor forma es parchacar SOLO lo que necesitas: `datetime.datetime.now`

**Ventajas:**
- ✓ El constructor `datetime.datetime()` sigue funcionando normalmente
- ✓ Código más limpio y específico
- ✓ No hay sorpresas con MagicMocks


In [ ]:
# ✅ VERSIÓN CORRECTA - Solución 1
class TestSaludoCorrecta(unittest.TestCase):

    @patch('datetime.datetime.now')  # ✅ Parcheamos SOLO datetime.now()
    def test_saludo_en_la_manana(self, mock_now):
        # datetime.datetime constructor sigue funcionando
        mock_now.return_value = datetime.datetime(2024, 1, 1, 9, 0, 0)
        self.assertEqual(obtener_saludo(), "Buenos días")

    @patch('datetime.datetime.now')
    def test_saludo_en_la_tarde(self, mock_now):
        mock_now.return_value = datetime.datetime(2024, 1, 1, 15, 0, 0)
        self.assertEqual(obtener_saludo(), "Buenas tardes")

    @patch('datetime.datetime.now')
    def test_saludo_en_la_noche(self, mock_now):
        mock_now.return_value = datetime.datetime(2024, 1, 1, 21, 0, 0)
        self.assertEqual(obtener_saludo(), "Buenas noches")


print("\n" + "=" * 60)
print("EJECUTANDO VERSIÓN CORRECTA...")
print("=" * 60)
suite = unittest.TestLoader().loadTestsFromTestCase(TestSaludoCorrecta)
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
print(f"\n✓ Pruebas exitosas: {resultado.testsRun}")


## ✅ SOLUCIÓN 2: Usar `spec` para Parchacar `datetime.datetime`

Si realmente necesitas parchacar todo `datetime.datetime` (caso menos común), usa `spec` para mantener el contrato del objeto:


In [ ]:
# ✅ SOLUCIÓN 2 - Alternativa: Parchacar todo pero permitir constructor real
class TestSaludoConSpec(unittest.TestCase):

    @patch('datetime.datetime', wraps=datetime.datetime)  # wraps permite usar el real
    def test_saludo_en_la_manana(self, mock_datetime):
        # Configurar solo lo que necesitamos parchacar
        mock_datetime.now.return_value = datetime.datetime(2024, 1, 1, 9, 0, 0)
        self.assertEqual(obtener_saludo(), "Buenos días")

    @patch('datetime.datetime', wraps=datetime.datetime)
    def test_saludo_en_la_tarde(self, mock_datetime):
        mock_datetime.now.return_value = datetime.datetime(2024, 1, 1, 15, 0, 0)
        self.assertEqual(obtener_saludo(), "Buenas tardes")

    @patch('datetime.datetime', wraps=datetime.datetime)
    def test_saludo_en_la_noche(self, mock_datetime):
        mock_datetime.now.return_value = datetime.datetime(2024, 1, 1, 21, 0, 0)
        self.assertEqual(obtener_saludo(), "Buenas noches")


print("\n" + "=" * 60)
print("SOLUCIÓN 2: Usando wraps (alternativa)")
print("=" * 60)
suite = unittest.TestLoader().loadTestsFromTestCase(TestSaludoConSpec)
resultado = unittest.TextTestRunner(verbosity=2).run(suite)
print(f"\n✓ Pruebas exitosas: {resultado.testsRun}")


## 📊 Comparación de Soluciones

| Enfoque | Código | Ventajas | Desventajas |
|---------|--------|----------|-------------|
| **❌ Malo** | `@patch('datetime.datetime')` | Parchaando muy amplio | TypeError con MagicMock |
| **✅ Mejor** | `@patch('datetime.datetime.now')` | Específico, limpio, simple | Solo funciona si usas `.now()` |
| **✅ Alternativa** | `@patch('datetime.datetime', wraps=...)` | Flexible, permite constructor | Más complejo de entender |

### 🎯 Recomendación
**Usa Solución 1 (`@patch('datetime.datetime.now')`)** en la mayoría de casos. Es la más clara y específica.

### 🔍 Lección Clave
Cuando usas `@patch()`, parchaeas lo que especifiques:
- `@patch('datetime.datetime')` → Reemplaza la clase COMPLETA ❌
- `@patch('datetime.datetime.now')` → Reemplaza SOLO el método ✅


In [ ]:
# 🎓 Demostración Visual del Problema
print("\n" + "=" * 70)
print("EXPLICACIÓN: ¿Por qué falla el patch('datetime.datetime')?")
print("=" * 70)

print("\n1️⃣ SIN patch - Comportamiento normal:")
print(f"   datetime.datetime.now() → {datetime.datetime.now()}")
print(f"   datetime.datetime.now().hour → {datetime.datetime.now().hour} (tipo: int)")

print("\n2️⃣ CON patch('datetime.datetime') - El problema:")
with patch('datetime.datetime') as mock_dt:
    print(f"   datetime.datetime.now() → {mock_dt.now()} (tipo: MagicMock)")
    print(f"   ¡Intentar comparar MagicMock < 12 causa TypeError!")

print("\n3️⃣ CON patch('datetime.datetime.now') - La solución:")
with patch('datetime.datetime.now') as mock_now:
    mock_now.return_value = datetime.datetime(2024, 1, 1, 9, 0, 0)
    print(f"   datetime.datetime.now() → {datetime.datetime.now()}")
    print(f"   datetime.datetime.now().hour → {datetime.datetime.now().hour} (tipo: int)")
    print(f"   ¡Funciona correctamente!")
